# Import packages and load data

In [ ]:
import pandas as pd
import numpy as np
import time
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs, rdMolDescriptors, MACCSkeys
from itertools import combinations
from mordred import Calculator, descriptors
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Input file with chemical names
file_path = '../validation_data.xlsx'
# Output path where we save the new excel files with likeness scores ready for ML training
output_path = './likeness_data/validation/'
# import data frame for pre-processing
df = pd.read_excel(file_path)
df

# Data pre-processing

In [ ]:
# Define mapping for 'solv_ratio' and apply if present
type_mapping = {'100/0': 1, '90/10': 0.9, '80/20': 0.8, '70/30': 0.7, '60/40': 0.6, '50/50': 0.5, '75/25': 0.75}
if 'solv_ratio' in df.columns:
    df['solv_ratio'] = df['solv_ratio'].replace(type_mapping)

# Define mapping for 'solv_ratio' and apply if present
type_mapping = {"100/0": 1, '80/20': 0.8, '60/40': 0.6, '50/50': 0.5, '40/60': 0.4, '20/80': 0.2}
if 'comp1_frac' in df.columns:
    df['comp1_frac'] = df['comp1_frac'].replace(type_mapping)


# Initialize the SMILES map
# Monomer unit
smiles_monomer_map = {
    "PS": "C=CC1=CC=CC=C1",               # Polystyrene
    "PEO": "C=CO",                 # Poly(ethylene oxide)
    "P2VP": "C=CC1=CC=CC=N1",    # Poly(2-vinylpyridine) 
    "P4VP": "C=CC1=CC=NC=C1",    # Poly(4-vinylpyridine)
    "PMMA": "CC(=C)C(=O)OC",    # Poly(methyl methacrylate)
    "PIB": "CC(=C)C",    # Poly(isobutylene)
    "PVA": "CC(=O)OC=C ",    # Poly(vinyl acetate) 
    "PCL": "OCCCCCC(=O)",    # Poly(ε-caprolacton) Linear form to better represent polymer structure
    "P2HEM": "CC(=C)C(=O)OCCO",    # Poly(2-Hydroxyethyl Methacrylate)
    "PtBuMA": "CC(=C)C(=O)OC(C)(C)C",    # Poly(t-butyl methacrylate)
    "PAN": "C=CC#N",    # Poly(acrylonitrile)
    "PtBuA": "CC(C)(C)OC(=O)C=C",    # Poly(t-butyl acrylate)
    "PnBuMA": "CCCCOC(=O)C(=C)C",    # Poly(n-butyl methacrylate)
    "Chloronaphthalene": "ClC1=CC=CC2=CC=CC=C12",
    "Methylnaphthalene": "CC1=CC=CC2=CC=CC=C12",
    "Toluene": "CC1=CC=CC=C1",
    "THF": "C1CCOC1",
    "CHCl3": "C(Cl)(Cl)Cl",
    "DMF": "CN(C)C=O"
}

# Repeat unit with polymerization points
smiles_poly_map = {
    "PS": "*CC(C1=CC=CC=C1)*",               # Polystyrene (repeat unit) example
    "PEO": "*CCO*",                 # Poly(ethylene oxide)
    "P2VP": "*CC(c1ccccn1)*",    # Poly(2-vinylpyridine) (monomer unit approx)
    "P4VP": "*CC(c1ccncc1)*",    # Poly(4-vinylpyridine)
    "PMMA": "*C(C)(C(=O)OC)*",    # Poly(methyl methacrylate)
    "PIB": "*C(C)(C)C*",    # Poly(isobutylene)
    "PVA": "*C(OC(=O)C)C*",    # Poly(vinyl acetate) 
    "PCL": "*OCCCCCC(=O)*",    # Poly(ε-caprolacton) Linear form to better represent polymer structure
    "P2HEM": "*C(C)(C*)C(=O)OCCO",    # Poly(2-Hydroxyethyl Methacrylate)
    "PtBuMA": "*CC(C)(*)C(=O)OC(C)(C)C",    # Poly(t-butyl methacrylate)
    "PAN": "*CC(C#N)*",    # Poly(acrylonitrile)
    "PtBuA": "*CC(*)C(=O)OC(C)(C)C",    # Poly(t-butyl acrylate)
    "PnBuMA": "CCCCOC(=O)C(C)(*)C*",    # Poly(n-butyl methacrylate)
    "Chloronaphthalene": "ClC1=CC=CC2=CC=CC=C12",
    "Methylnaphthalene": "CC1=CC=CC2=CC=CC=C12",
    "Toluene": "CC1=CC=CC=C1",
    "THF": "C1CCOC1",
    "CHCl3": "C(Cl)(Cl)Cl",
    "DMF": "CN(C)C=O"
}

# Repeat unit
smiles_repeat_map = {
    "PS": "CC(C1=CC=CC=C1)",               # Polystyrene (repeat unit) example
    "PEO": "CCO",                 # Poly(ethylene oxide)
    "P2VP": "CC(c1ccccn1)",    # Poly(2-vinylpyridine) (monomer unit approx)
    "P4VP": "CC(c1ccncc1)",    # Poly(4-vinylpyridine)
    "PMMA": "C(C)(C(=O)OC)",    # Poly(methyl methacrylate)
    "PIB": "C(C)(C)C",    # Poly(isobutylene)
    "PVA": "C(OC(=O)C)C",    # Poly(vinyl acetate) 
    "PCL": "OCCCCCC(=O)",    # Poly(ε-caprolacton) Linear form to better represent polymer structure
    "P2HEM": "C(C)(C)C(=O)OCCO",    # Poly(2-Hydroxyethyl Methacrylate)
    "PtBuMA": "CC(C)C(=O)OC(C)(C)C",    # Poly(t-butyl methacrylate)
    "PAN": "CC(C#N)",    # Poly(acrylonitrile)
    "PtBuA": "CCC(=O)OC(C)(C)C",    # Poly(t-butyl acrylate)
    "PnBuMA": "CCCCOC(=O)C(C)C",    # Poly(n-butyl methacrylate)
    "Chloronaphthalene": "ClC1=CC=CC2=CC=CC=C12",
    "Methylnaphthalene": "CC1=CC=CC2=CC=CC=C12",
    "Toluene": "CC1=CC=CC=C1",
    "THF": "C1CCOC1",
    "CHCl3": "C(Cl)(Cl)Cl",
    "DMF": "CN(C)C=O"
}

# Columnds with chemical information that needs to be converted to SMILES
chem_cols = [
    "comp1_blockA_type", "comp1_blockB_type",
    "comp2_blockA_type", "comp2_blockB_type",
    "add_type", "solv_1", "solv_2"
]

In [ ]:
# Create a master smiles map containing the 3 maps
smiles_maps = [
    ("poly",   smiles_poly_map),
    ("repeat", smiles_repeat_map),
    ("monomer", smiles_monomer_map)
]

df_results = {}  # store each DataFrame by a key

# For each SMILES map, create a new df that replaces chemical names with SMILES structures
for name, smap in smiles_maps:
    # Work on a copy so originals aren’t clobbered
    temp_df = df.copy()
    for col in chem_cols:
        if col in temp_df.columns:
            temp_df[col] = temp_df[col].map(smap).fillna(temp_df[col])
    df_results[name] = temp_df # df_results contains multiple dfs, one for each SMILES map

In [ ]:
# Mordred calculator
mordred_calc = Calculator(descriptors, ignore_3D=True)

# Computes the similarity of two SMILES structures using a variety of fingerprinting/descriptor methods
def compute_similarity(smiles1, smiles2, method="morgan", **kwargs):

    # Ensure there are no empty SMILES being passed, do not write an output if this is the case
    if pd.isna(smiles1) or pd.isna(smiles2) or smiles1 == "" or smiles2 == "":
        return None
    mol1, mol2 = Chem.MolFromSmiles(smiles1), Chem.MolFromSmiles(smiles2)
    if mol1 is None or mol2 is None:
        return None

    method = method.lower()

    # Morgan fingerprinting method with Tanimoto similarity
    if method == "morgan":
        radius = kwargs.get("radius", 2)
        nBits = kwargs.get("nBits", 2048)
        fp1 = AllChem.GetMorganFingerprintAsBitVect(mol1, radius, nBits=nBits)
        fp2 = AllChem.GetMorganFingerprintAsBitVect(mol2, radius, nBits=nBits)
        return DataStructs.TanimotoSimilarity(fp1, fp2)
        
    # RDkit_path fingerprinting method with Tanimoto similarity
    elif method == "rdkit_path":
        nBits = kwargs.get("nBits", 2048)
        minPath = kwargs.get("minPath", 1)
        maxPath = kwargs.get("maxPath", 7)
        fp1 = Chem.RDKFingerprint(mol1, fpSize=nBits, minPath=minPath, maxPath=maxPath)
        fp2 = Chem.RDKFingerprint(mol2, fpSize=nBits, minPath=minPath, maxPath=maxPath)
        return DataStructs.TanimotoSimilarity(fp1, fp2)
        
    # MACCS fingerprinting method with Tanimoto similarity
    elif method == "maccs":
        fp1 = MACCSkeys.GenMACCSKeys(mol1)
        fp2 = MACCSkeys.GenMACCSKeys(mol2)
        return DataStructs.TanimotoSimilarity(fp1, fp2)
        
    # RDkit_physchem descriptor method with Cosine similarity
    elif method == "rdkit_physchem":
        from rdkit.Chem import Descriptors
        desc_list = [f for name, f in Descriptors.descList]
        # Store the RDkit_physchem desciptors in an array
        v1 = np.array([f(mol1) for f in desc_list], dtype=float)
        v2 = np.array([f(mol2) for f in desc_list], dtype=float)
        return float(cosine_similarity([v1], [v2])[0][0])
        
    # Mordred descriptor method with Cosine similarity (This one takes FOREVER)
    elif method == "mordred":
        try:
            d1 = mordred_calc(mol1).fill_missing(0)
            d2 = mordred_calc(mol2).fill_missing(0)
        except Exception:
            return None
        # Convert the Mordred desciptors to an array
        v1 = np.array(list(d1.values()), dtype=float)
        v2 = np.array(list(d2.values()), dtype=float)
        if v1.size == 0 or v2.size == 0:
            return None
        return float(cosine_similarity([v1], [v2])[0][0])

    else:
        raise ValueError(f"Unknown method: {method}") # Incase a method is passed incorrectly to the function

# Comb through each row and pass every possible pair of SMILES structures to the similarity calcuator
def calculate_likeness_row(row, method):
    results = {}
    for col1, col2 in col_pairs:
        pair_name = f"{col1}_{col2}"
        likeness = compute_similarity(row.get(col1), row.get(col2), method=method)
        results[f"{pair_name}_likeness"] = likeness
        results[f"{pair_name}_exists"] = 0 if likeness is None else 1
    return pd.Series(results)

# The resulting df is messy and difficult to parse, this function reorganizes the df, specific to this df (Not generalizable)
def column_reoganizer(df):
    # Rearrange the columns for easier analysis down the line
    cols = list(df.columns)
    # First column is related to the sample ids (Keep for troubleshooting)
    sample_ids = cols[0:1]
    # Out generated likeness descriptors fall at the end of the sheet
    likeness_descriptors = cols[17:min(59,len(cols))]
    # Processing conditions are jumbled up due to original formatting
    mw_descriptors = [cols[i] for i in [2, 4, 6, 8] if i < len(cols)]
    processing_desciptors_1 = cols[9:min(11,len(cols))]
    processing_desciptors_2 = cols[12:min(13,len(cols))]
    processing_descriptors = list(processing_desciptors_1) + list(processing_desciptors_2) + list(mw_descriptors)
    # Outputs are at the end of the original df
    outputs = cols[13:min(15,len(cols))]
    # Combine all of these into one list of column names
    new_cols = list(sample_ids) + list(likeness_descriptors) + processing_descriptors + list(outputs)
    # Create a new organized df
    df_organized = df.loc[:, new_cols]  # loc ensures we’re selecting by labels
    return df_organized

In [ ]:
%%time

# Similarity methods we are using
similarity_methods = ["morgan", "rdkit_path", "maccs", "rdkit_physchem", "mordred"]
# Generate all column pairs that may contain SMILES structures
col_pairs = list(combinations(chem_cols, 2))
df_likeness = {}
for name, smap in smiles_maps:
    for method in similarity_methods:
        # Continuous descriptor methods cannot handle the wildcard present in the 'poly' SMILES
        if "poly" in name.lower() and method in ["rdkit_physchem", "mordred"]:
            continue
        start_time = time.time()
        # Compute likeness for this method
        likeness_df = df_results[name].apply(calculate_likeness_row, axis=1, method=method)

        # Combine original with likeness scores
        temp_df = pd.concat([df_results[name], likeness_df], axis=1)
        
        # Reorganize
        temp_df = column_reoganizer(temp_df)

        # Identify all columns except the last two (Selects all of our input columns)
        cols_to_fill = temp_df.columns[:-2]      

        # Fill NaN values in these columns with 0 (These are cases where the likeness pair does not exist)
        temp_df[cols_to_fill] =  temp_df[cols_to_fill].fillna(0)

        df_likeness_key = f"{name}_{method}"
        df_likeness[df_likeness_key] = temp_df

        # Save to distinct Excel
        df_likeness[df_likeness_key].to_excel(f"{output_path}{name}_{method}_validation_subset.xlsx", index=False)
        
        end_time = time.time()

        elapsed = (end_time - start_time)/60
        print(f"Time for {df_likeness_key}: {elapsed:.2f} minutes")